
# 🌍 Lab 4: การประมวลผลข้อมูลด้วย NumPy และ GeoPandas
## วิชา GE 234 Basic Programming for Geographers

### 🎯 **วัตถุประสงค์**
1. เข้าใจการใช้ **NumPy** สำหรับการประมวลผลข้อมูลทางภูมิศาสตร์ เช่น ข้อมูล Raster และพิกัด
2. ใช้ **GeoPandas** ในการจัดการและวิเคราะห์ข้อมูลเวกเตอร์ เช่น **Shapefile**
3. สามารถดำเนินการทางสถิติกับข้อมูลพิกัดและชั้นข้อมูลทางภูมิศาสตร์ได้
4. สามารถใช้ NumPy และ GeoPandas ร่วมกันเพื่อวิเคราะห์ข้อมูลได้

---

## 🔹 ตัวอย่างที่ 1: ใช้ NumPy คำนวณข้อมูล NDVI จากภาพ Raster


In [ ]:

import numpy as np

# สร้างข้อมูลตัวอย่างสำหรับภาพ NDVI (Normalized Difference Vegetation Index)
nir = np.array([[0.7, 0.8, 0.6], [0.9, 0.5, 0.3], [0.4, 0.7, 0.2]])
red = np.array([[0.3, 0.4, 0.2], [0.5, 0.3, 0.1], [0.2, 0.3, 0.1]])

# คำนวณค่า NDVI
ndvi = (nir - red) / (nir + red)

print("ค่า NDVI:")
print(ndvi)


ค่า NDVI:
[[0.4        0.33333333 0.5       ]
 [0.28571429 0.25       0.5       ]
 [0.33333333 0.4        0.33333333]]



## 🔹 ตัวอย่างที่ 2: ใช้ NumPy คำนวณค่าเฉลี่ยและค่ามากสุดของ NDVI


In [ ]:

print(f"ค่าเฉลี่ย NDVI: {np.mean(ndvi):.2f}")
print(f"ค่า NDVI สูงสุด: {np.max(ndvi):.2f}")
print(f"ค่า NDVI ต่ำสุด: {np.min(ndvi):.2f}")


ค่าเฉลี่ย NDVI: 0.37
ค่า NDVI สูงสุด: 0.50
ค่า NDVI ต่ำสุด: 0.25



## 🔹 ตัวอย่างที่ 3: ใช้ GeoPandas โหลดและวิเคราะห์ข้อมูล Shapefile


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import geopandas as gpd

# โหลดข้อมูลชั้นข้อมูลจังหวัดของประเทศไทยจาก Google Drive
gdf = gpd.read_file("/content/drive/MyDrive/Colab Notebooks/tha_admbnda_adm1_rtsd_20190221")

# แสดงข้อมูล 5 แถวแรก
print(gdf.head())

   Shape_Leng  Shape_Area        ADM1_EN        ADM1_TH ADM1_PCODE ADM1_REF  \
0    3.927244    0.275313  Amnat Charoen     อำนาจเจริญ       TH37     None   
1    1.739908    0.079210      Ang Thong        อ่างทอง       TH15     None   
2    2.417227    0.131339        Bangkok  กรุงเทพมหานคร       TH10     None   
3    4.414998    0.340784      Bueng Kan         บึงกาฬ       TH38     None   
4    8.701860    0.844537       Buri Ram      บุรีรัมย์       TH31     None   

  ADM1ALT1EN ADM1ALT2EN ADM1ALT1TH ADM1ALT2TH   ADM0_EN    ADM0_TH ADM0_PCODE  \
0       None       None       None       None  Thailand  ประเทศไทย         TH   
1       None       None       None       None  Thailand  ประเทศไทย         TH   
2       None       None       None       None  Thailand  ประเทศไทย         TH   
3       None       None       None       None  Thailand  ประเทศไทย         TH   
4       None       None       None       None  Thailand  ประเทศไทย         TH   

        date    validOn validTo  \
0 2


## 🔹 ตัวอย่างที่ 4: คำนวณพื้นที่ของแต่ละจังหวัด


In [37]:

# ตรวจสอบค่า CRS (Coordinate Reference System)
print(gdf.crs)

# คำนวณพื้นที่ของแต่ละจังหวัด (หน่วยเป็นตารางเมตร)
gdf["area_sqkm"] = gdf.geometry.area / 1e6

# แสดงพื้นที่จังหวัด 5 อันดับแรกที่ใหญ่ที่สุด
print(gdf.nlargest(5, "area_sqkm")[["ADM1_TH", "area_sqkm"]])


EPSG:4326
        ADM1_TH  area_sqkm
9     เชียงใหม่   0.000002
28   นครราชสีมา   0.000002
15    กาญจนบุรี   0.000002
68          ตาก   0.000001
71  อุบลราชธานี   0.000001


/tmp/ipykernel_2312/3624066604.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["area_sqkm"] = gdf.geometry.area / 1e6



## 🔹 ตัวอย่างที่ 5: ใช้ GeoPandas ทำ Spatial Join


In [19]:

# โหลดข้อมูลชั้นข้อมูลอำเภอ
gdf_districts = gpd.read_file("/content/drive/MyDrive/Colab Notebooks/thailand_province_amphoe (1)")

# ทำ Spatial Join ระหว่างอำเภอกับจังหวัด
gdf_joined = gpd.sjoin(gdf_districts, gdf, how="inner", predicate="within")

# แสดงตัวอย่างข้อมูลที่เชื่อมโยงกัน
print(gdf_joined.head())


   Shape_Leng_left  Shape_Area_left      ADM2_EN  ADM2_TH ADM2_PCODE  \
0         0.085417         0.000450  Phra Nakhon   พระนคร     TH1001   
1         0.134132         0.000950        Dusit    ดุสิต     TH1002   
2         0.676342         0.019859    Nong Chok  หนองจอก     TH1003   
3         0.085886         0.000337     Bang Rak   บางรัก     TH1004   
4         0.301722         0.003415    Bang Khen   บางเขน     TH1005   

  ADM1_EN_left   ADM1_TH_left ADM1_PCODE_left ADM0_EN_left ADM0_TH_left  ...  \
0      Bangkok  กรุงเทพมหานคร            TH10     Thailand    ประเทศไทย  ...   
1      Bangkok  กรุงเทพมหานคร            TH10     Thailand    ประเทศไทย  ...   
2      Bangkok  กรุงเทพมหานคร            TH10     Thailand    ประเทศไทย  ...   
3      Bangkok  กรุงเทพมหานคร            TH10     Thailand    ประเทศไทย  ...   
4      Bangkok  กรุงเทพมหานคร            TH10     Thailand    ประเทศไทย  ...   

  ADM1ALT1EN ADM1ALT2EN  ADM1ALT1TH  ADM1ALT2TH  ADM0_EN_right ADM0_TH_right  \
0     


# 📝 **กิจกรรมในแลป**

1. **แบบฝึกหัด 1**: ใช้ NumPy คำนวณค่า **Mean, Max, Min** ของค่า NDVI ในอาร์เรย์ที่สร้างขึ้นเอง
2. **แบบฝึกหัด 2**: ใช้ GeoPandas โหลด **Shapefile** ของจังหวัด และคำนวณพื้นที่ของแต่ละจังหวัด
3. **แบบฝึกหัด 3**: ใช้ GeoPandas ทำ **Spatial Join** ระหว่างข้อมูลจังหวัดและอำเภอ
4. **แบบฝึกหัด 4**: ใช้ NumPy และ GeoPandas ร่วมกันเพื่อหาข้อมูลจังหวัดที่มี NDVI เฉลี่ยสูงสุด


**แบบฝึกหัด 1** ใช้ NumPy คำนวณค่า Mean, Max, Min ของค่า NDVI ในอาร์เรย์ที่สร้างขึ้นเอง

In [ ]:
print(f"ค่าเฉลี่ย NDVI: {np.mean(ndvi):.2f}")
print(f"ค่า NDVI สูงสุด: {np.max(ndvi):.2f}")
print(f"ค่า NDVI ต่ำสุด: {np.min(ndvi):.2f}")

ค่าเฉลี่ย NDVI: 0.37
ค่า NDVI สูงสุด: 0.50
ค่า NDVI ต่ำสุด: 0.25


In [6]:
import numpy as np

# อาร์เรย์ NDVI ที่สร้างคิดขึ้นเอง
my_ndvi_array = np.array([0.5, 0.6, 0.7, 0.3, 0.4, 0.8, 0.9, 0.2, 0.1])

print("อาร์เรย์ NDVI ที่สร้างขึ้นเอง:")
print(my_ndvi_array)

# จากนั้นนำอาร์เรย์นี้ไปคำนวณ Mean, Max, Min
print(f"ค่าเฉลี่ย NDVI: {np.mean(my_ndvi_array):.2f}")
print(f"ค่า NDVI สูงสุด : {np.max(my_ndvi_array):.2f}")
print(f"ค่า NDVI ต่ำสุด : {np.min(my_ndvi_array):.2f}")

อาร์เรย์ NDVI ที่สร้างขึ้นเอง:
[0.5 0.6 0.7 0.3 0.4 0.8 0.9 0.2 0.1]
ค่าเฉลี่ย NDVI: 0.50
ค่า NDVI สูงสุด : 0.90
ค่า NDVI ต่ำสุด : 0.10


**แบบฝึกหัด 2** ใช้ GeoPandas โหลด Shapefile ของจังหวัด และคำนวณพื้นที่ของแต่ละจังหวัด

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [58]:
import geopandas as gpd

# โหลดข้อมูลชั้นข้อมูลจังหวัดของประเทศไทยจาก Google Drive
gdf_provinces = gpd.read_file("/content/drive/MyDrive/Colab Notebooks/tha_admbnda_adm1_rtsd_20190221")

# แปลง CRS เป็นระบบพิกัดฉาย (Projected CRS) ที่เหมาะสมสำหรับการคำนวณพื้นที่
gdf_provinces_projected = gdf_provinces.to_crs(epsg=3857)

# คำนวณพื้นที่ของแต่ละจังหวัด (หน่วยเป็นตารางเมตร จากนั้นแปลงเป็นตารางกิโลเมตร)
gdf_provinces_projected["area_sqkm"] = gdf_provinces_projected.geometry.area / 1e6

# แสดงพื้นที่จังหวัด 10 อันดับแรกที่ใหญ่ที่สุด
print("10 จังหวัดแรกที่มีพื้นที่มากที่สุด:")
display(gdf_provinces_projected.nlargest(10, "area_sqkm")[["ADM1_TH", "area_sqkm"]])

10 จังหวัดแรกที่มีพื้นที่มากที่สุด:


,ADM1_TH,area_sqkm
9,เชียงใหม่,24880.095257
28,นครราชสีมา,22317.931970
15,กาญจนบุรี,20891.294974
68,ตาก,18946.947972
71,อุบลราชธานี,16720.834730
22,แม่ฮ่องสอน,14331.639196
18,ลำปาง,13943.694560
7,ชัยภูมิ,13735.960029
31,น่าน,13684.956167
41,เพชรบูรณ์,13522.636263


In [8]:
import geopandas as gpd

# อ่านไฟล์
gdf = gpd.read_file('/content/drive/MyDrive/Colab Notebooks/tha_admbnda_adm1_rtsd_20190221')

# ตรวจสอบค่า CRS (Coordinate Reference System)
# print(f"CRS เดิมของข้อมูล: {gdf.crs}")

# แปลง CRS เป็นระบบพิกัดฉาย (Projected CRS) ที่เหมาะสมสำหรับการคำนวณพื้นที่
# ใช้ EPSG:3857 (Web Mercator) ซึ่งเป็น Projected CRS
# สำหรับประเทศไทย ใช้ UTM Zone 47N (EPSG:32647) หรือ 48N (EPSG:32648)
gdf_projected = gdf.to_crs(epsg=3857)
# print(f"CRS หลังแปลง: {gdf_projected.crs}")

# คำนวณพื้นที่ของแต่ละจังหวัด (แปลงหน่วยเป็นตารางเมตร จากนั้นแปลงเป็นตารางกิโลเมตร) #เพื่่อให้มีข้อมูลตรงในไฟล์
gdf_projected["area_sqkm"] = gdf_projected.geometry.area / 1e6

# แสดงพื้นที่จังหวัด 10 อันดับแรกที่ใหญ่ที่สุด
print(gdf_projected.nlargest(10, "area_sqkm")[["ADM1_TH", "area_sqkm"]])

        ADM1_TH     area_sqkm
9     เชียงใหม่  24880.095257
28   นครราชสีมา  22317.931970
15    กาญจนบุรี  20891.294974
68          ตาก  18946.947972
71  อุบลราชธานี  16720.834730
22   แม่ฮ่องสอน  14331.639196
18        ลำปาง  13943.694560
7       ชัยภูมิ  13735.960029
31         น่าน  13684.956167
41    เพชรบูรณ์  13522.636263


**แบบฝึกหัด 3** ใช้ GeoPandas ทำ Spatial Join ระหว่างข้อมูลจังหวัดและอำเภอ

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
# โหลดข้อมูลชั้นข้อมูลอำเภอ
gdf_districts = gpd.read_file("/content/drive/MyDrive/Colab Notebooks/thailand_province_amphoe (1)")
# ทำ Spatial Join ระหว่างอำเภอกับจังหวัด
gdf_joined = gpd.sjoin(gdf_districts, gdf, how="inner", predicate="within")

# แสดงตัวอย่างข้อมูลที่เชื่อมโยงกัน
print(gdf_joined.head())

# เครดิตข้อมูลจาก ไฟล์ thailand_gis

   Shape_Leng_left  Shape_Area_left      ADM2_EN  ADM2_TH ADM2_PCODE  \
0         0.085417         0.000450  Phra Nakhon   พระนคร     TH1001   
1         0.134132         0.000950        Dusit    ดุสิต     TH1002   
2         0.676342         0.019859    Nong Chok  หนองจอก     TH1003   
3         0.085886         0.000337     Bang Rak   บางรัก     TH1004   
4         0.301722         0.003415    Bang Khen   บางเขน     TH1005   

  ADM1_EN_left   ADM1_TH_left ADM1_PCODE_left ADM0_EN_left ADM0_TH_left  ...  \
0      Bangkok  กรุงเทพมหานคร            TH10     Thailand    ประเทศไทย  ...   
1      Bangkok  กรุงเทพมหานคร            TH10     Thailand    ประเทศไทย  ...   
2      Bangkok  กรุงเทพมหานคร            TH10     Thailand    ประเทศไทย  ...   
3      Bangkok  กรุงเทพมหานคร            TH10     Thailand    ประเทศไทย  ...   
4      Bangkok  กรุงเทพมหานคร            TH10     Thailand    ประเทศไทย  ...   

  ADM1ALT1EN ADM1ALT2EN  ADM1ALT1TH  ADM1ALT2TH  ADM0_EN_right ADM0_TH_right  \
0     

In [12]:
# ตรวจสอบว่ามีอำเภอในจังหวัดอื่นๆ อยู่ใน gdf_joined หรือไม่
print("จังหวัดทั้งหมดที่พบหลังจากการ Spatial Join:")
print(gdf_joined['ADM1_TH_left'].unique())


จังหวัดทั้งหมดที่พบหลังจากการ Spatial Join:
['กรุงเทพมหานคร' 'สมุทรปราการ' 'นนทบุรี' 'ปทุมธานี' 'พระนครศรีอยุธยา'
 'อ่างทอง' 'ลพบุรี' 'สิงห์บุรี' 'ชัยนาท' 'สระบุรี' 'ชลบุรี' 'ระยอง'
 'จันทบุรี' 'ตราด' 'ฉะเชิงเทรา' 'ปราจีนบุรี' 'นครนายก' 'สระแก้ว'
 'นครราชสีมา' 'บุรีรัมย์' 'สุรินทร์' 'ศรีสะเกษ' 'อุบลราชธานี' 'ยโสธร'
 'ชัยภูมิ' 'อำนาจเจริญ' 'บึงกาฬ' 'หนองบัวลำภู' 'ขอนแก่น' 'อุดรธานี' 'เลย'
 'หนองคาย' 'มหาสารคาม' 'ร้อยเอ็ด' 'กาฬสินธุ์' 'สกลนคร' 'นครพนม' 'มุกดาหาร'
 'เชียงใหม่' 'ลำพูน' 'ลำปาง' 'อุตรดิตถ์' 'แพร่' 'น่าน' 'พะเยา' 'เชียงราย'
 'แม่ฮ่องสอน' 'นครสวรรค์' 'อุทัยธานี' 'กำแพงเพชร' 'ตาก' 'สุโขทัย'
 'พิษณุโลก' 'พิจิตร' 'เพชรบูรณ์' 'ราชบุรี' 'กาญจนบุรี' 'สุพรรณบุรี'
 'นครปฐม' 'สมุทรสาคร' 'สมุทรสงคราม' 'เพชรบุรี' 'ประจวบคีรีขันธ์'
 'นครศรีธรรมราช' 'กระบี่' 'พังงา' 'ภูเก็ต' 'สุราษฎร์ธานี' 'ระนอง' 'ชุมพร'
 'สงขลา' 'สตูล' 'ตรัง' 'พัทลุง' 'ปัตตานี' 'ยะลา' 'นราธิวาส']


In [18]:
print("\nตัวอย่างอำเภอในจังหวัดเชียงใหม่:")
display(gdf_joined[gdf_joined['ADM1_TH_left'] == 'เชียงใหม่'].head(10))


ตัวอย่างอำเภอในจังหวัดเชียงใหม่:


,Shape_Leng_left,Shape_Area_left,ADM2_EN,ADM2_TH,ADM2_PCODE,ADM1_EN_left,ADM1_TH_left,ADM1_PCODE_left,ADM0_EN_left,ADM0_TH_left,...,ADM1ALT1EN,ADM1ALT2EN,ADM1ALT1TH,ADM1ALT2TH,ADM0_EN_right,ADM0_TH_right,ADM0_PCODE_right,date,validOn,validTo
519,0.823859,0.015495,Mueang Chiang Mai,เมืองเชียงใหม่,TH5001,Chiang Mai,เชียงใหม่,TH50,Thailand,ประเทศไทย,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT
520,1.885059,0.095307,Chom Thong,จอมทอง,TH5002,Chiang Mai,เชียงใหม่,TH50,Thailand,ประเทศไทย,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT
521,3.242044,0.234037,Mae Chaem,แม่แจ่ม,TH5003,Chiang Mai,เชียงใหม่,TH50,Thailand,ประเทศไทย,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT
522,2.645922,0.188725,Chiang Dao,เชียงดาว,TH5004,Chiang Mai,เชียงใหม่,TH50,Thailand,ประเทศไทย,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT
523,1.606988,0.055995,Doi Saket,ดอยสะเก็ด,TH5005,Chiang Mai,เชียงใหม่,TH50,Thailand,ประเทศไทย,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT
524,2.293483,0.121298,Mae Taeng,แม่แตง,TH5006,Chiang Mai,เชียงใหม่,TH50,Thailand,ประเทศไทย,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT
525,1.109831,0.038957,Mae Rim,แม่ริม,TH5007,Chiang Mai,เชียงใหม่,TH50,Thailand,ประเทศไทย,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT
526,1.797231,0.086388,Samoeng,สะเมิง,TH5008,Chiang Mai,เชียงใหม่,TH50,Thailand,ประเทศไทย,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT
527,1.566140,0.070454,Fang,ฝาง,TH5009,Chiang Mai,เชียงใหม่,TH50,Thailand,ประเทศไทย,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT
528,1.764846,0.069061,Mae Ai,แม่อาย,TH5010,Chiang Mai,เชียงใหม่,TH50,Thailand,ประเทศไทย,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT


In [17]:
import pandas as pd
# เซลล์นี้ทำเพื่อเช็คว่าข้อมูลที่ได้มีจังหวัดอื่นๆด้วยมั้ย เพราะที่รันได้ 10 อันดับแรกเจอแต่ของ กทม. ค่ะ
# ตั้งค่าให้ Pandas แสดงคอลัมน์ทั้งหมด ไม่จำกัดจำนวน
pd.set_option('display.max_columns', None)

# แสดง 125 แถวแรกของข้อมูลที่เชื่อมโยงกัน (สามารถเปลี่ยนเป็นจำนวนอื่นได้ค่ะ)
print(gdf_joined.head(125))

# หากต้องการดูข้อมูลทั้งหมด
# print(gdf_joined)

# คืนค่าการแสดงผลคอลัมน์กลับเป็นค่าเริ่มต้น (ไม่บังคับ)
pd.reset_option('display.max_columns')


     Shape_Leng_left  Shape_Area_left      ADM2_EN   ADM2_TH ADM2_PCODE  \
0           0.085417         0.000450  Phra Nakhon    พระนคร     TH1001   
1           0.134132         0.000950        Dusit     ดุสิต     TH1002   
2           0.676342         0.019859    Nong Chok   หนองจอก     TH1003   
3           0.085886         0.000337     Bang Rak    บางรัก     TH1004   
4           0.301722         0.003415    Bang Khen    บางเขน     TH1005   
..               ...              ...          ...       ...        ...   
120         0.730908         0.015710  Wihan Daeng  วิหารแดง     TH1904   
121         0.446812         0.007079   Nong Saeng   หนองแซง     TH1905   
122         0.732822         0.012934       Ban Mo   บ้านหมอ     TH1906   
123         0.333597         0.004787     Don Phut    ดอนพุด     TH1907   
124         0.548732         0.007730     Nong Don   หนองโดน     TH1908   

    ADM1_EN_left   ADM1_TH_left ADM1_PCODE_left ADM0_EN_left ADM0_TH_left  \
0        Bangkok  กรุง

In [16]:
print("ตัวอย่างการจับคู่อำเภอกับจังหวัด:")
display(gdf_joined[['ADM2_TH', 'ADM1_TH_left']].head(125))


ตัวอย่างการจับคู่อำเภอกับจังหวัด:


,ADM2_TH,ADM1_TH_left
0,พระนคร,กรุงเทพมหานคร
1,ดุสิต,กรุงเทพมหานคร
2,หนองจอก,กรุงเทพมหานคร
3,บางรัก,กรุงเทพมหานคร
4,บางเขน,กรุงเทพมหานคร
...,...,...
120,วิหารแดง,สระบุรี
121,หนองแซง,สระบุรี
122,บ้านหมอ,สระบุรี
123,ดอนพุด,สระบุรี


**แบบฝึกหัด 4**: ใช้ NumPy และ GeoPandas ร่วมกันเพื่อหาข้อมูลจังหวัดที่มี NDVI เฉลี่ยสูงสุด (จากไฟล์อำเภอ)

In [93]:
import numpy as np
import geopandas as gpd

if 'gdf' not in locals():
    gdf = gpd.read_file("/content/drive/MyDrive/Colab Notebooks/thailand_province_amphoe (1)")

# หาจำนวนจังหวัด
num_provinces = len(gdf)
print(f"จำนวนจังหวัดใน GeoDataFrame: {num_provinces}")

จำนวนจังหวัดใน GeoDataFrame: 77


In [102]:
display(province_avg_ndvi_ex4.sort_values(by='district_ndvi', ascending=False))

,ADM1_TH,district_ndvi
62,หนองคาย,0.631155
42,ระยอง,0.620256
15,นครนายก,0.598601
66,อุตรดิตถ์,0.570854
4,กำแพงเพชร,0.561718
...,...,...
51,สตูล,0.375631
16,นครปฐม,0.366016
75,แพร่,0.349550
28,ปราจีนบุรี,0.348152


In [103]:
print(f"จังหวัดที่มีค่าเฉลี่ย NDVI สูงสุดคือ: {highest_avg_ndvi_province_data_ex4['ADM1_TH']} ด้วยค่า NDVI เฉลี่ย {highest_avg_ndvi_province_data_ex4['district_ndvi']:.2f}")
display(highest_avg_ndvi_province_data_ex4)

จังหวัดที่มีค่าเฉลี่ย NDVI สูงสุดคือ: หนองคาย ด้วยค่า NDVI เฉลี่ย 0.63


,62
ADM1_TH,หนองคาย
district_ndvi,0.631155
